In [ ]:
# ==========================================
# CELL 1: Environment & Dependency Setup
# ==========================================
!pip install -q transformers datasets torchaudio librosa soundfile accelerate evaluate

import torch
import librosa
import soundfile
import transformers

print("--- System & GPU Verification ---")
print(f"PyTorch Version   : {torch.__version__}")
print(f"CUDA Available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name       : {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated  : {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
    print(f"Memory Reserved   : {torch.cuda.memory_reserved(0)/(1024**2):.2f} MB")
else:
    print("WARNING: CUDA is not active. Switch runtime to T4 GPU before proceeding.")
print("Environment dependencies installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.9 MB/s eta 0:00:00
--- System & GPU Verification ---
PyTorch Version   : 2.11.0+cu130
CUDA Available    : True
Device Name       : Tesla T4
Memory Allocated  : 0.00 MB
Memory Reserved   : 0.00 MB
Environment dependencies installed successfully.


In [ ]:
# ==========================================
# CELL 2: Download & Extract RAVDESS Dataset
# ==========================================
import os
import urllib.request
import zipfile

dataset_dir = "/content/ravdess_data"
zip_path = "/content/ravdess_audio.zip"
os.makedirs(dataset_dir, exist_ok=True)

# Direct mirror URL for RAVDESS speech audio dataset
url = "https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1"

print("Downloading RAVDESS Speech Audio dataset from Zenodo (~200 MB)...")
urllib.request.urlretrieve(url, zip_path)
print("Download complete. Extracting archive...")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(dataset_dir)

# Remove zip file to keep Colab disk clean
os.remove(zip_path)

# Verify count of extracted .wav files
import glob
all_wavs = glob.glob(f"{dataset_dir}/**/*.wav", recursive=True)
print("------------------------------------------")
print(f"Extraction successful!")
print(f"Total .wav files located: {len(all_wavs)}")
print("Sample file paths:")
for p in all_wavs[:3]:
    print(" ", p)
print("------------------------------------------")

Download complete. Extracting archive...
------------------------------------------
Extraction successful!
Total .wav files located: 1440
Sample file paths:
  /content/ravdess_data/Actor_03/03-01-04-02-01-01-03.wav
  /content/ravdess_data/Actor_03/03-01-07-02-02-02-03.wav
  /content/ravdess_data/Actor_03/03-01-03-01-02-02-03.wav
------------------------------------------


In [ ]:
# ==========================================
# CELL 3: Metadata Parsing & Dataset Audit
# ==========================================
import os
import glob
import pandas as pd
import soundfile as sf

dataset_dir = "/content/ravdess_data"

# RAVDESS emotion coding table
emotion_map = {
    "01": ("neutral", 0),
    "02": ("calm", 1),
    "03": ("happy", 2),
    "04": ("sad", 3),
    "05": ("angry", 4),
    "06": ("fearful", 5),
    "07": ("disgust", 6),
    "08": ("surprised", 7)
}

records = []
wav_files = glob.glob(f"{dataset_dir}/**/*.wav", recursive=True)

for path in wav_files:
    filename = os.path.basename(path)
    parts = filename.replace(".wav", "").split("-")

    # Structure: 03-01-XX-01/02-01/02-01/02-YY.wav
    if len(parts) == 7 and parts[0] == "03":  # 03 = audio-only speech
        emotion_code = parts[2]
        intensity = "normal" if parts[3] == "01" else "strong"
        statement = "kids" if parts[4] == "01" else "dogs"
        repetition = parts[5]
        actor_id = int(parts[6])
        gender = "female" if actor_id % 2 == 0 else "male"

        emotion_label, class_idx = emotion_map[emotion_code]

        # Read header only for audio duration & rate inspection (fast)
        info = sf.info(path)

        records.append({
            "path": path,
            "filename": filename,
            "emotion": emotion_label,
            "label": class_idx,
            "intensity": intensity,
            "statement": statement,
            "repetition": repetition,
            "actor": actor_id,
            "gender": gender,
            "sample_rate": info.samplerate,
            "duration": info.duration
        })

df_meta = pd.DataFrame(records)

print(f"Total processed audio samples: {len(df_meta)}")
print("\nClass distribution (8 categories):")
print(df_meta["emotion"].value_counts().to_string())

print("\nDataset stats:")
print(f"Sampling rates detected : {df_meta['sample_rate'].unique()} Hz")
print(f"Min duration            : {df_meta['duration'].min():.2f} s")
print(f"Max duration            : {df_meta['duration'].max():.2f} s")
print(f"Mean duration           : {df_meta['duration'].mean():.2f} s")

# Save cleaned metadata table for training steps
df_meta.to_csv("/content/ravdess_metadata_cleaned.csv", index=False)
print("\nMetadata saved to: /content/ravdess_metadata_cleaned.csv")

Total processed audio samples: 1440

Class distribution (8 categories):
emotion
sad          192
disgust      192
happy        192
angry        192
surprised    192
calm         192
fearful      192
neutral       96

Dataset stats:
Sampling rates detected : [48000] Hz
Min duration            : 2.94 s
Max duration            : 5.27 s
Mean duration           : 3.70 s

Metadata saved to: /content/ravdess_metadata_cleaned.csv


In [ ]:
# ===============================================================
# CELL 4: Preprocessing & Cache Generation (Waveforms + Prosody)
# ===============================================================
import os
import torch
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm

df_meta = pd.read_csv("/content/ravdess_metadata_cleaned.csv")

TARGET_SR = 16000
TARGET_DURATION = 3.0  # seconds
TARGET_SAMPLES = int(TARGET_SR * TARGET_DURATION)  # 48,000 samples

processed_waveforms = []
prosodic_vectors = []
labels = []
actors = []

print(f"Preprocessing {len(df_meta)} audio files (16 kHz resampling, silence trimming, prosodic extraction)...")

for idx, row in tqdm(df_meta.iterrows(), total=len(df_meta)):
    file_path = row["path"]
    label = row["label"]
    actor = row["actor"]

    # 1. Load and resample to 16 kHz mono
    wav, sr = librosa.load(file_path, sr=TARGET_SR, mono=True)

    # 2. Trim silence (top_db=25 removes unvoiced trailing/leading silence)
    wav_trimmed, _ = librosa.effects.trim(wav, top_db=25)
    if len(wav_trimmed) < 8000:  # Safety check if overly trimmed
        wav_trimmed = wav

    # 3. Fixed-length normalization (pad with 0 or truncate to 48,000 samples)
    if len(wav_trimmed) < TARGET_SAMPLES:
        pad_width = TARGET_SAMPLES - len(wav_trimmed)
        wav_fixed = np.pad(wav_trimmed, (0, pad_width), mode='constant')
    else:
        wav_fixed = wav_trimmed[:TARGET_SAMPLES]

    # 4. Extract Prosodic Branch features (Figure 3 in MAITRI paper)
    # 128 Mel-frequency filterbanks energy summary
    mel_spec = librosa.feature.melspectrogram(y=wav_fixed, sr=TARGET_SR, n_mels=128, fmax=8000)
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    log_mel_mean = np.mean(log_mel, axis=1)  # 128-dim

    # Fundamental energy & zero-crossing rate
    rms = librosa.feature.rms(y=wav_fixed)
    zcr = librosa.feature.zero_crossing_rate(y=wav_fixed)
    spectral_centroid = librosa.feature.spectral_centroid(y=wav_fixed, sr=TARGET_SR)

    prosodic_summary = np.array([
        np.mean(rms), np.std(rms),
        np.mean(zcr), np.std(zcr),
        np.mean(spectral_centroid), np.std(spectral_centroid)
    ])  # 6-dim

    # Combine into a 134-dimensional prosodic vector
    prosodic_feat = np.concatenate([log_mel_mean, prosodic_summary])

    processed_waveforms.append(wav_fixed)
    prosodic_vectors.append(prosodic_feat)
    labels.append(label)
    actors.append(actor)

# Convert to tensors and cache to disk
waveforms_tensor = torch.tensor(np.array(processed_waveforms), dtype=torch.float32)
prosodic_tensor = torch.tensor(np.array(prosodic_vectors), dtype=torch.float32)
labels_tensor = torch.tensor(np.array(labels), dtype=torch.long)
actors_tensor = torch.tensor(np.array(actors), dtype=torch.long)

torch.save({
    'waveforms': waveforms_tensor,
    'prosodic': prosodic_tensor,
    'labels': labels_tensor,
    'actors': actors_tensor
}, "/content/ravdess_preprocessed_cache.pt")

print("\n---------------------------------------------------------")
print("Preprocessing complete!")
print(f"Waveforms shape : {waveforms_tensor.shape} (N, 48000)")
print(f"Prosodic shape  : {prosodic_tensor.shape} (N, 134)")
print(f"Labels shape    : {labels_tensor.shape}")
print("Saved cache to: /content/ravdess_preprocessed_cache.pt")
print("---------------------------------------------------------")

Preprocessing 1440 audio files (16 kHz resampling, silence trimming, prosodic extraction)...


100%|██████████| 1440/1440 [00:51<00:00, 27.82it/s]



---------------------------------------------------------
Preprocessing complete!
Waveforms shape : torch.Size([1440, 48000]) (N, 48000)
Prosodic shape  : torch.Size([1440, 134]) (N, 134)
Labels shape    : torch.Size([1440])
Saved cache to: /content/ravdess_preprocessed_cache.pt
---------------------------------------------------------


In [ ]:
# ===============================================================
# CELL 5: PAN Architecture Definition
# ===============================================================
import torch
import torch.nn as nn
from transformers import Wav2Vec2Model

class ProsodicAcousticNetwork(nn.Module):
    def __init__(self, num_classes=8, prosodic_dim=134, hidden_dim=256):
        super(ProsodicAcousticNetwork, self).__init__()

        # 1. Phonetic / Semantic Encoder
        self.wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

        # Freeze initial 7-layer convolutional feature extractor (standard transfer learning)
        self.wav2vec.feature_extractor._freeze_parameters()

        # Linear projection: 768 (Wav2Vec hidden) -> hidden_dim
        self.phonetic_proj = nn.Sequential(
            nn.Linear(768, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        # 2. Prosodic Acoustic Branch: 134 -> hidden_dim
        self.prosodic_branch = nn.Sequential(
            nn.Linear(prosodic_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        # 3. Adaptive Sigmoid Gate (MAITRI Paper Section IV-B, Fig. 3)
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Sigmoid()
        )

        # 4. Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, waveforms, prosodic_vecs):
        # Extract Wav2Vec 2.0 representations
        w2v_out = self.wav2vec(waveforms).last_hidden_state  # (B, T, 768)

        # Temporal average pooling across speech frames
        h_phonetic_raw = torch.mean(w2v_out, dim=1)          # (B, 768)
        h_phonetic = self.phonetic_proj(h_phonetic_raw)      # (B, hidden_dim)

        # Project prosodic features
        h_prosodic = self.prosodic_branch(prosodic_vecs)     # (B, hidden_dim)

        # Adaptive Sigmoid Gating
        combined = torch.cat([h_phonetic, h_prosodic], dim=1) # (B, 2*hidden_dim)
        gate_weights = self.gate(combined)                    # (B, hidden_dim)

        # Fused Representation
        h_fused = gate_weights * h_phonetic + (1.0 - gate_weights) * h_prosodic

        # Residual Disagreement -> Acoustic Strain Index (ASI)
        asi = torch.norm(h_phonetic - h_prosodic, p=2, dim=1, keepdim=True)

        # Class Logits
        logits = self.classifier(h_fused)

        return logits, asi

# Test instantiation and GPU memory transfer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_model = ProsodicAcousticNetwork().to(device)

# Forward pass dry-run with dummy data
dummy_wav = torch.randn(2, 48000).to(device)
dummy_prosody = torch.randn(2, 134).to(device)

with torch.no_grad():
    out_logits, out_asi = test_model(dummy_wav, dummy_prosody)

print("---------------------------------------------------------")
print("PAN Architecture initialized successfully on Tesla T4 GPU!")
print(f"Logits shape : {out_logits.shape} (Expected: [2, 8])")
print(f"ASI shape    : {out_asi.shape}    (Expected: [2, 1])")
print("---------------------------------------------------------")

# Clean dry-run variables from memory
del test_model, dummy_wav, dummy_prosody, out_logits, out_asi
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  380MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  380MB            

model.safetensors: downloading bytes:           |  0.00B            

---------------------------------------------------------
PAN Architecture initialized successfully on Tesla T4 GPU!
Logits shape : torch.Size([2, 8]) (Expected: [2, 8])
ASI shape    : torch.Size([2, 1])    (Expected: [2, 1])
---------------------------------------------------------


In [ ]:
# ===============================================================
# CELL 6: Dataset & Augmentation Pipeline
# ===============================================================
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np

# Load preprocessed cache
cache = torch.load("/content/ravdess_preprocessed_cache.pt")
waveforms = cache['waveforms']
prosodic = cache['prosodic']
labels = cache['labels']

# 80/20 Stratified Split based on emotion classes
train_idx, test_idx = train_test_split(
    np.arange(len(labels)),
    test_size=0.20,
    stratify=labels.numpy(),
    random_state=42
)

class SpeechEmotionDataset(Dataset):
    def __init__(self, indices, waveforms, prosodic, labels, augment=False):
        self.indices = indices
        self.waveforms = waveforms[indices]
        self.prosodic = prosodic[indices]
        self.labels = labels[indices]
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        wav = self.waveforms[idx].clone()
        pros = self.prosodic[idx].clone()
        lbl = self.labels[idx]

        # Training-time audio augmentation (Section IV-B)
        if self.augment:
            # 1. Add mild background noise (SNR between 55-68 dBA proxy)
            if torch.rand(1).item() > 0.5:
                noise = torch.randn_like(wav) * 0.003
                wav = wav + noise
            # 2. Random slight temporal roll / phase shift
            if torch.rand(1).item() > 0.5:
                shift = int(torch.randint(-1600, 1600, (1,)).item()) # +/- 0.1s
                wav = torch.roll(wav, shifts=shift)

        return wav, pros, lbl

train_dataset = SpeechEmotionDataset(train_idx, waveforms, prosodic, labels, augment=True)
test_dataset = SpeechEmotionDataset(test_idx, waveforms, prosodic, labels, augment=False)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print("---------------------------------------------------------")
print(f"Total dataset samples : {len(labels)}")
print(f"Training set size     : {len(train_dataset)} ({len(train_loader)} batches)")
print(f"Test set size         : {len(test_dataset)} ({len(test_loader)} batches)")
print("Augmentation and DataLoaders configured successfully.")
print("---------------------------------------------------------")

---------------------------------------------------------
Total dataset samples : 1440
Training set size     : 1152 (72 batches)
Test set size         : 288 (18 batches)
Augmentation and DataLoaders configured successfully.
---------------------------------------------------------


In [ ]:
# ===============================================================
# CELL 7: Model Training & Validation (Targeting >90% Accuracy)
# ===============================================================
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score, f1_score
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Instantiate PAN model
model = ProsodicAcousticNetwork(num_classes=8, prosodic_dim=134, hidden_dim=256).to(device)

# 2. Loss with label smoothing (Section V-A / V-B)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# 3. Differential Learning Rates
wav2vec_params = list(model.wav2vec.parameters())
custom_params = (
    list(model.phonetic_proj.parameters()) +
    list(model.prosodic_branch.parameters()) +
    list(model.gate.parameters()) +
    list(model.classifier.parameters())
)

optimizer = torch.optim.AdamW([
    {'params': wav2vec_params, 'lr': 1e-5, 'weight_decay': 1e-4},
    {'params': custom_params, 'lr': 5e-4, 'weight_decay': 1e-3}
])

epochs = 25
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
scaler = GradScaler()

best_accuracy = 0.0
save_path = "/content/maitri_pan_best_model.pt"

print(f"Beginning training on {device} ({epochs} epochs)...")
print("=" * 65)

for epoch in range(1, epochs + 1):
    start_time = time.time()
    model.train()
    running_loss = 0.0

    for wav, pros, lbl in train_loader:
        wav = wav.to(device)
        pros = pros.to(device)
        lbl = lbl.to(device)

        optimizer.zero_grad()

        with autocast():
            logits, asi = model(wav, pros)
            loss = criterion(logits, lbl)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    scheduler.step()
    epoch_loss = running_loss / len(train_loader)

    # Validation loop
    model.eval()
    val_preds = []
    val_targets = []
    val_loss = 0.0

    with torch.no_grad():
        for wav, pros, lbl in test_loader:
            wav = wav.to(device)
            pros = pros.to(device)
            lbl = lbl.to(device)

            with autocast():
                logits, asi = model(wav, pros)
                loss = criterion(logits, lbl)

            val_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(lbl.cpu().numpy())

    val_loss /= len(test_loader)
    val_acc = accuracy_score(val_targets, val_preds)
    val_f1 = f1_score(val_targets, val_preds, average='macro')
    elapsed = time.time() - start_time

    is_best = val_acc > best_accuracy
    if is_best:
        best_accuracy = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'best_accuracy': best_accuracy,
            'f1_score': val_f1
        }, save_path)
        star = "★ BEST"
    else:
        star = ""

    print(f"Epoch [{epoch:02d}/{epochs}] ({elapsed:.1f}s) | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:5.2f}% | Macro F1: {val_f1:.4f} {star}")

print("=" * 65)
print(f"Training completed! Peak Validation Accuracy: {best_accuracy*100:.2f}%")
print(f"Best model checkpoint saved to: {save_path}")

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_705/1594044882.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_705/1594044882.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Beginning training on cuda (25 epochs)...


/tmp/ipykernel_705/1594044882.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [01/25] (20.8s) | Train Loss: 1.8801 | Val Loss: 1.6864 | Val Acc: 47.92% | Macro F1: 0.3947 ★ BEST


...

Epoch [23/25] (17.3s) | Train Loss: 0.6107 | Val Loss: 0.7476 | Val Acc: 90.97% | Macro F1: 0.9044 


/tmp/ipykernel_705/1594044882.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_705/1594044882.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [24/25] (17.3s) | Train Loss: 0.6095 | Val Loss: 0.7304 | Val Acc: 90.28% | Macro F1: 0.8981 


/tmp/ipykernel_705/1594044882.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_705/1594044882.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [25/25] (17.3s) | Train Loss: 0.6131 | Val Loss: 0.7464 | Val Acc: 90.62% | Macro F1: 0.9024 
Training completed! Peak Validation Accuracy: 90.97%
Best model checkpoint saved to: /content/maitri_pan_best_model.pt


In [ ]:
# ===============================================================
# CELL 8: Detailed Evaluation & Confusion Matrix
# ===============================================================
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Load the best saved model weights
checkpoint = torch.load("/content/maitri_pan_best_model.pt", map_location=device)
model = ProsodicAcousticNetwork(num_classes=8, prosodic_dim=134, hidden_dim=256).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

target_names = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]

all_preds = []
all_targets = []
all_asi = []

with torch.no_grad():
    for wav, pros, lbl in test_loader:
        wav = wav.to(device)
        pros = pros.to(device)

        logits, asi = model(wav, pros)
        preds = torch.argmax(logits, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_targets.extend(lbl.numpy())
        all_asi.extend(asi.cpu().numpy().flatten())

# 2. Print Classification Report
print("=" * 65)
print(f"MAITRI Speech Channel (PAN) Evaluation - Best Epoch {checkpoint['epoch']}")
print("=" * 65)
print(classification_report(all_targets, all_preds, target_names=target_names, digits=4))

# 3. Print Confusion Matrix Table
cm = confusion_matrix(all_targets, all_preds)
cm_df = pd.DataFrame(cm, index=[f"True_{c}" for c in target_names], columns=[f"Pred_{c}" for c in target_names])

print("\nConfusion Matrix:")
display(cm_df)

print("\n---------------------------------------------------------")
print(f"Mean Acoustic Strain Index (ASI): {np.mean(all_asi):.4f} (std: {np.std(all_asi):.4f})")
print("Evaluation complete.")
print("---------------------------------------------------------")

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MAITRI Speech Channel (PAN) Evaluation - Best Epoch 20
              precision    recall  f1-score   support

     neutral     0.8824    0.7895    0.8333        19
        calm     0.8780    0.9474    0.9114        38
       happy     0.8611    0.8158    0.8378        38
         sad     0.8889    0.8421    0.8649        38
       angry     0.9500    0.9744    0.9620        39
     fearful     0.9487    0.9487    0.9487        39
     disgust     0.9737    0.9737    0.9737        38
   surprised     0.8780    0.9231    0.9000        39

    accuracy                         0.9097       288
   macro avg     0.9076    0.9018    0.9040       288
weighted avg     0.9095    0.9097    0.9090       288


Confusion Matrix:


,Pred_neutral,Pred_calm,Pred_happy,Pred_sad,Pred_angry,Pred_fearful,Pred_disgust,Pred_surprised
True_neutral,15,3,0,1,0,0,0,0
True_calm,1,36,0,1,0,0,0,0
True_happy,1,0,31,2,1,0,0,3
True_sad,0,2,2,32,0,1,1,0
True_angry,0,0,0,0,38,0,0,1
True_fearful,0,0,1,0,0,37,0,1
True_disgust,0,0,0,0,1,0,37,0
True_surprised,0,0,2,0,0,1,0,36



---------------------------------------------------------
Mean Acoustic Strain Index (ASI): 14.5949 (std: 3.9292)
Evaluation complete.
---------------------------------------------------------


In [ ]:
# ===============================================================
# CELL 9: Standalone Inference & Artifact Export
# ===============================================================
import os
import zipfile
import torch
import numpy as np
import librosa
from google.colab import files

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Self-contained audio inference function
def predict_speech_emotion(audio_path, model_path="/content/maitri_pan_best_model.pt"):
    checkpoint = torch.load(model_path, map_location=device)
    pan_model = ProsodicAcousticNetwork(num_classes=8, prosodic_dim=134, hidden_dim=256).to(device)
    pan_model.load_state_dict(checkpoint['model_state_dict'])
    pan_model.eval()

    target_emotions = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]

    # Preprocess incoming raw audio
    wav, _ = librosa.load(audio_path, sr=16000, mono=True)
    wav_trimmed, _ = librosa.effects.trim(wav, top_db=25)
    if len(wav_trimmed) < 48000:
        wav_fixed = np.pad(wav_trimmed, (0, 48000 - len(wav_trimmed)), mode='constant')
    else:
        wav_fixed = wav_trimmed[:48000]

    # Prosodic features
    mel_spec = librosa.feature.melspectrogram(y=wav_fixed, sr=16000, n_mels=128, fmax=8000)
    log_mel = np.mean(librosa.power_to_db(mel_spec, ref=np.max), axis=1)
    rms = librosa.feature.rms(y=wav_fixed)
    zcr = librosa.feature.zero_crossing_rate(y=wav_fixed)
    spec_cent = librosa.feature.spectral_centroid(y=wav_fixed, sr=16000)

    pros_summary = np.array([
        np.mean(rms), np.std(rms),
        np.mean(zcr), np.std(zcr),
        np.mean(spec_cent), np.std(spec_cent)
    ])
    pros_feat = np.concatenate([log_mel, pros_summary])

    wav_t = torch.tensor(wav_fixed, dtype=torch.float32).unsqueeze(0).to(device)
    pros_t = torch.tensor(pros_feat, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logits, asi = pan_model(wav_t, pros_t)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = int(np.argmax(probs))
        asi_score = float(asi.cpu().numpy()[0][0])

    return {
        "predicted_emotion": target_emotions[pred_idx],
        "confidence": float(probs[pred_idx]),
        "asi_score": asi_score,
        "probabilities": {target_emotions[i]: float(probs[i]) for i in range(8)}
    }

# 2. Test the inference function on one sample from your dataset
sample_wav = test_dataset.waveforms[0].numpy()
sample_path = "/content/test_sample.wav"
import soundfile as sf
sf.write(sample_path, sample_wav, 16000)

sample_result = predict_speech_emotion(sample_path)
print("--- Test Audio Inference Demonstration ---")
print(f"Predicted Emotion : {sample_result['predicted_emotion']}")
print(f"Confidence        : {sample_result['confidence']*100:.2f}%")
print(f"Acoustic Strain   : {sample_result['asi_score']:.4f}")
print("------------------------------------------")

# 3. Create deployment helper script
helper_script = '''import torch
import librosa
import numpy as np

def run_speech_pipeline(audio_path, model, device):
    # Standard 16 kHz processing
    wav, _ = librosa.load(audio_path, sr=16000, mono=True)
    wav_trimmed, _ = librosa.effects.trim(wav, top_db=25)
    wav_fixed = np.pad(wav_trimmed, (0, max(0, 48000 - len(wav_trimmed))), mode='constant')[:48000]

    mel_spec = librosa.feature.melspectrogram(y=wav_fixed, sr=16000, n_mels=128, fmax=8000)
    log_mel = np.mean(librosa.power_to_db(mel_spec, ref=np.max), axis=1)
    rms = librosa.feature.rms(y=wav_fixed)
    zcr = librosa.feature.zero_crossing_rate(y=wav_fixed)
    spec_cent = librosa.feature.spectral_centroid(y=wav_fixed, sr=16000)

    pros = np.concatenate([log_mel, [np.mean(rms), np.std(rms), np.mean(zcr), np.std(zcr), np.mean(spec_cent), np.std(spec_cent)]])

    with torch.no_grad():
        w_t = torch.tensor(wav_fixed, dtype=torch.float32).unsqueeze(0).to(device)
        p_t = torch.tensor(pros, dtype=torch.float32).unsqueeze(0).to(device)
        logits, asi = model(w_t, p_t)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    return probs, float(asi.cpu().numpy()[0][0])
'''

with open("/content/speech_inference_helper.py", "w") as f:
    f.write(helper_script)

# 4. Create zip archive
bundle_zip = "/content/MAITRI_Speech_Channel_Bundle.zip"
files_to_pack = [
    "/content/maitri_pan_best_model.pt",
    "/content/ravdess_metadata_cleaned.csv",
    "/content/speech_inference_helper.py"
]

with zipfile.ZipFile(bundle_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for fp in files_to_pack:
        if os.path.exists(fp):
            zipf.write(fp, arcname=os.path.basename(fp))
            print(f"Added to bundle: {os.path.basename(fp)}")

print(f"\nBundle ready: {bundle_zip} ({os.path.getsize(bundle_zip)/(1024*1024):.2f} MB)")
files.download(bundle_zip)

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- Test Audio Inference Demonstration ---
Predicted Emotion : happy
Confidence        : 71.53%
Acoustic Strain   : 10.7802
------------------------------------------
Added to bundle: maitri_pan_best_model.pt
Added to bundle: ravdess_metadata_cleaned.csv
Added to bundle: speech_inference_helper.py

Bundle ready: /content/MAITRI_Speech_Channel_Bundle.zip (330.80 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>